<a href="https://colab.research.google.com/github/Prathamesh-exe/AI-ML-Colabs-and-Gists/blob/main/rlmbig.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip -q install openai
from openai import OpenAI
from collections import Counter
import json
import re
import time
import uuid

In [3]:
OPENCODE_API_KEY = "oc_sk_NICETRY"

SESSION_ID = str(uuid.uuid4())

client = OpenAI(
    api_key=OPENCODE_API_KEY,
    base_url="https://opencode.ai/zen/go/v1",
    default_headers={
        "x-opencode-session": SESSION_ID,
        "User-Agent": "rlm-openstack-log-investigator/1.0"
    }
)

MODEL = "deepseek-v4-flash"

In [4]:
LOG_FILES = [
    "/content/openstack_normal1.log",
    "/content/openstack_normal2.log",
    "/content/openstack_abnormal.log",
]

RAW_LOGS = []

for file_path in LOG_FILES:
    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    RAW_LOGS.extend(lines)

    print(f"{file_path}")
    print(f"  Lines: {len(lines):,}")

print(f"\nTotal lines: {len(RAW_LOGS):,}")

/content/openstack_normal1.log
  Lines: 52,312
/content/openstack_normal2.log
  Lines: 137,074
/content/openstack_abnormal.log
  Lines: 18,434

Total lines: 207,820


In [5]:
LOG_PATTERN = re.compile(
    r"""
    ^(?P<source>\S+)
    \s+
    (?P<timestamp>\d{4}-\d{2}-\d{2}
                     \s+
                     \d{2}:\d{2}:\d{2}\.\d+)
    \s+
    (?P<pid>\d+)
    \s+
    (?P<level>DEBUG|INFO|WARNING|WARN|ERROR|CRITICAL)
    \s+
    (?P<service>\S+)
    \s+
    (?P<message>.*)
    $
    """,
    re.VERBOSE
)


def parse_log_line(line, line_number, source_file):
    raw = line.strip()

    match = LOG_PATTERN.match(raw)

    if not match:
        return {
            "line_number": line_number,
            "source_file": source_file,
            "raw": raw,
            "parse_error": True
        }

    data = match.groupdict()

    data["line_number"] = line_number
    data["source_file"] = source_file
    data["raw"] = raw
    data["parse_error"] = False

    return data

In [6]:
PARSED_LOGS = []

global_line_number = 0

for file_path in LOG_FILES:

    with open(file_path, "r", encoding="utf-8", errors="replace") as f:

        for line in f:
            global_line_number += 1

            record = parse_log_line(
                line,
                global_line_number,
                file_path.split("/")[-1]
            )

            PARSED_LOGS.append(record)

print(f"Total records: {len(PARSED_LOGS):,}")

parse_errors = sum(
    1 for record in PARSED_LOGS
    if record["parse_error"]
)

print(f"Parse errors: {parse_errors:,}")

Total records: 207,820
Parse errors: 8


In [57]:
INITIAL_WINDOW = 3000

def inspect_initial_window(limit=3000, sample_size=20):
    records = PARSED_LOGS[:limit]

    levels = Counter(
        x["level"]
        for x in records
        if not x["parse_error"]
    )

    services = Counter(
        x["service"]
        for x in records
        if not x["parse_error"]
    )

    messages = Counter(
        x["message"]
        for x in records
        if not x["parse_error"]
    )

    suspicious = [
        x for x in records
        if x.get("level") in {"WARNING", "ERROR", "CRITICAL"}
    ]

    return {
        "inspected_lines": len(records),
        "range": f"1-{len(records)}",
        "levels": dict(levels),
        "top_services": dict(services.most_common(10)),
        "top_messages": [
            {
                "message": msg,
                "count": count
            }
            for msg, count in messages.most_common(10)
        ],
        "representative_suspicious_logs": suspicious[:sample_size]
    }

def search_messages(query, max_results=20):
    max_results = min(max_results, 20)

    query_lower = query.lower()

    matches = [
        x for x in PARSED_LOGS
        if query_lower in x.get("message", "").lower()
    ]

    return {
        "query": query,
        "total_matches": len(matches),
        "results": matches[:max_results]
    }

def group_messages(level=None, service=None, top_n=20):
    records = PARSED_LOGS

    if level:
        records = [
            x for x in records
            if x.get("level") == level.upper()
        ]

    if service:
        records = [
            x for x in records
            if service.lower() in x.get("service", "").lower()
        ]

    groups = Counter(
        x.get("message", "")
        for x in records
    )

    return {
        "filters": {
            "level": level,
            "service": service
        },
        "total_records": len(records),
        "groups": [
            {
                "message": message,
                "count": count
            }
            for message, count in groups.most_common(top_n)
        ]
    }

def service_summary(service, top_n=20):
    records = [
        x for x in PARSED_LOGS
        if service.lower() in x.get("service", "").lower()
    ]

    levels = Counter(x["level"] for x in records)
    messages = Counter(x["message"] for x in records)

    return {
        "service": service,
        "total_records": len(records),
        "log_levels": dict(levels),
        "top_messages": [
            {
                "message": message,
                "count": count
            }
            for message, count in messages.most_common(top_n)
        ]
    }

def get_context(line_number, radius=3):
    radius = min(radius, 5)

    start = max(1, line_number - radius)
    end = min(len(PARSED_LOGS), line_number + radius)

    return {
        "target_line": line_number,
        "range": {
            "start": start,
            "end": end
        },
        "results": PARSED_LOGS[start - 1:end]
    }

def severity_summary():
    levels = Counter(
        x["level"]
        for x in PARSED_LOGS
        if not x["parse_error"]
    )

    return dict(levels)

In [58]:
TOOLS = {
    "inspect_initial_window": inspect_initial_window,
    "search_messages": search_messages,
    "group_messages": group_messages,
    "service_summary": service_summary,
    "get_context": get_context,
}

In [55]:
TOOL_DEFINITIONS = [
    {
        "type": "function",
        "function": {
            "name": "inspect_initial_window",
            "description": (
                "Inspect the first small window of the OpenStack dataset. "
                "Use this as the initial investigation step."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "limit": {
                        "type": "integer",
                        "description": "Number of initial log lines to inspect.",
                        "default": 3000
                    }
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_messages",
            "description": (
                "Search the entire dataset for a specific message pattern. "
                "Returns only matching records, not the full dataset."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string"
                    },
                    "max_results": {
                        "type": "integer",
                        "default": 20
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "group_messages",
            "description": (
                "Find recurring message patterns across the dataset, "
                "optionally filtered by severity or service."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "level": {
                        "type": "string"
                    },
                    "service": {
                        "type": "string"
                    },
                    "top_n": {
                        "type": "integer",
                        "default": 20
                    }
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "service_summary",
            "description": "Analyze a specific OpenStack service.",
            "parameters": {
                "type": "object",
                "properties": {
                    "service": {
                        "type": "string"
                    },
                    "top_n": {
                        "type": "integer",
                        "default": 20
                    }
                },
                "required": ["service"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_context",
            "description": "Retrieve log lines surrounding a specific line number.",
            "parameters": {
                "type": "object",
                "properties": {
                    "line_number": {
                        "type": "integer"
                    },
                    "radius": {
                        "type": "integer",
                        "default": 3
                    }
                },
                "required": ["line_number"]
            }
        }
    }
]

In [66]:
SYSTEM_PROMPT = """
You are an autonomous Recursive Language Model investigating
OpenStack infrastructure logs.

The complete dataset contains more than 200,000 log messages and
exists outside your context window in a Python environment.

The dataset MUST NOT be loaded into your context.

Investigation strategy:

1. ALWAYS begin with inspect_initial_window.

2. inspect_initial_window examines the first 3,000 lines internally,
   but returns only aggregated statistics and a small number of
   representative records.

3. Do not request or expect thousands of raw log records.

4. After identifying a suspicious pattern, use search_messages
   to search the complete dataset for that specific pattern.

5. search_messages returns only a small number of representative
   matches plus the total match count.

6. Use group_messages to measure recurring patterns.

7. Use service_summary when a specific service becomes relevant.

8. Use get_context only for important individual line numbers.

9. Keep every investigation step targeted.

10. Never request large result sets.

11. Follow evidence recursively:

    observation
        ↓
    targeted query
        ↓
    evidence
        ↓
    next query

12. The fact that Python can access the complete dataset does NOT
    mean the complete dataset should be returned to you.

13. Do not expose chain-of-thought or internal reasoning.

14. Do not invent facts.

15. Stop when enough evidence exists to answer the question.

FINAL RESPONSE RULES:

- Be extremely concise.
- Maximum 3 bullets.
- Each bullet must contain:
  Issue — Service — Frequency — Key evidence.
- Keep each bullet to one sentence.
- Use short resource IDs/paths only when they add value.
- Do not explain the investigation.
- Do not repeat the same evidence.
- Finish with one short overall observation.
- Do not expose chain-of-thought.
"""

In [62]:
def compact_result(tool_name, result):

    if tool_name == "inspect_initial_window":
        return (
            f"inspected {result['range']} | "
            f"levels={result['levels']} | "
            f"services={list(result['top_services'].keys())[:3]}"
        )

    if tool_name == "search_messages":
        return (
            f"'{result['query']}' → "
            f"{result['total_matches']:,} matches"
        )

    if tool_name == "group_messages":
        groups = result.get("groups", [])[:3]

        if not groups:
            return "no matching groups"

        return " | ".join(
            f"{x['count']:,}× {x['message'][:55]}"
            for x in groups
        )

    if tool_name == "service_summary":
        return (
            f"{result['service']} → "
            f"{result['total_records']:,} records"
        )

    if tool_name == "get_context":
        r = result["range"]

        return (
            f"lines {r['start']:,}-{r['end']:,}"
        )

    return "result returned"

In [67]:
def run_rlm(question, max_steps=10):

    print("\n" + "=" * 64)
    print("RECURSIVE LANGUAGE MODEL")
    print("=" * 64)
    print(f"Question: {question}")
    print(f"Model:    {MODEL}")
    print(f"Dataset:  {len(PARSED_LOGS):,} log lines")
    print("=" * 64)

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": question
        }
    ]

    trace = []

    for step in range(1, max_steps + 1):

        started = time.time()

        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=TOOL_DEFINITIONS,
            tool_choice="auto"
        )

        elapsed = time.time() - started
        message = response.choices[0].message

        if not message.tool_calls:

            print(
                f"\nSTEP {step}  →  FINAL ANSWER "
                f"({elapsed:.1f}s)"
            )

            return message.content, trace

        tool_names = [
            call.function.name
            for call in message.tool_calls
        ]

        print(
            f"\nSTEP {step}  →  "
            f"{' + '.join(tool_names)}"
        )

        assistant_data = {
            "role": "assistant",
            "content": message.content
        }

        assistant_data["tool_calls"] = [
            {
                "id": call.id,
                "type": "function",
                "function": {
                    "name": call.function.name,
                    "arguments": call.function.arguments
                }
            }
            for call in message.tool_calls
        ]

        messages.append(assistant_data)

        for call in message.tool_calls:

            tool_name = call.function.name

            try:
                arguments = json.loads(
                    call.function.arguments or "{}"
                )

                result = TOOLS[tool_name](**arguments)

            except Exception as e:
                result = {"error": str(e)}

            summary = compact_result(
                tool_name,
                result
            )

            print(f"          ↳ {summary}")

            trace.append({
                "step": step,
                "tool": tool_name,
                "arguments": arguments,
                "summary": summary
            })

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(
                    result,
                    default=str
                )
            })

    print("\nMAX STEPS REACHED")

    return None, trace

In [68]:
question = """
What are the top recurring issues in these OpenStack logs,
including their services, resources, frequencies, and related events?
"""

result, trace = run_rlm(
    question,
    max_steps=10
)


RECURSIVE LANGUAGE MODEL
Question: 
What are the top recurring issues in these OpenStack logs,
including their services, resources, frequencies, and related events?

Model:    deepseek-v4-flash
Dataset:  207,820 log lines

STEP 1  →  inspect_initial_window
          ↳ inspected 1-3000 | levels={'INFO': 2952, 'WARNING': 48} | services=['nova.osapi_compute.wsgi.server', 'nova.virt.libvirt.imagecache', 'nova.compute.manager']

STEP 2  →  group_messages + group_messages + group_messages
          ↳ 3,059× [req-addc1839-2ed5-4778-b57e-5854eb7b8b09 - - - - -] Un | 45× [req-addc1839-2ed5-4778-b57e-5854eb7b8b09 - - - - -] Wh | 1× [req-addc1839-2ed5-4778-b57e-5854eb7b8b09 - - - - -] co
          ↳ 8× [req-addc1839-2ed5-4778-b57e-5854eb7b8b09 - - - - -] Er | 8× Traceback (most recent call last): | 8× File "/usr/lib/python2.7/dist-packages/oslo_service/per
          ↳ 12,043× [req-addc1839-2ed5-4778-b57e-5854eb7b8b09 - - - - -] im | 12,043× [req-addc1839-2ed5-4778-b57e-5854eb7b8b09 - - - - -] im

In [70]:
from IPython.display import display, Markdown

# Group actual tool calls by RLM step
steps = {}

for item in trace:
    step = item["step"]
    steps.setdefault(step, []).append(item)

trace_rows = []

for step, items in steps.items():
    tools = " + ".join(f"`{x['tool']}`" for x in items)

    # Keep only a short signal from the actual results
    signals = []
    for x in items:
        s = x["summary"].replace("\n", " ")

        if "→" in s:
            s = s.split("→")[-1].strip()

        signals.append(s[:55])

    evidence = " · ".join(signals[:2])

    trace_rows.append(
        f"| **{step}** | {tools} | {evidence} |"
    )

trace_table = "\n".join(trace_rows)

display(Markdown(f"""
# RLM Execution

**Question:** {question.strip()}

**DeepSeek V4 Flash** · **207,820 logs** · **Python REPL**

| Step | Tools | Evidence |
|:---:|---|---|
{trace_table}

**Flow:**
**Question → DeepSeek → Python → Evidence → DeepSeek → ↻ → Final Answer**

> **207K+ logs stay outside the model context.**

### Final Answer

{result}
"""))


# RLM Execution

**Question:** What are the top recurring issues in these OpenStack logs,
including their services, resources, frequencies, and related events?

**DeepSeek V4 Flash** · **207,820 logs** · **Python REPL**

| Step | Tools | Evidence |
|:---:|---|---|
| **1** | `inspect_initial_window` | inspected 1-3000 | levels={'INFO': 2952, 'WARNING': 48} |
| **2** | `group_messages` + `group_messages` + `group_messages` | 3,059× [req-addc1839-2ed5-4778-b57e-5854eb7b8b09 - - -  · 8× [req-addc1839-2ed5-4778-b57e-5854eb7b8b09 - - - - -] |
| **3** | `search_messages` + `search_messages` + `search_messages` + `search_messages` | 12 matches · 2 matches |
| **4** | `get_context` + `service_summary` | lines 27,174-27,184 · 24,784 records |
| **5** | `get_context` | lines 27,185-27,195 |
| **6** | `get_context` | lines 27,191-27,201 |
| **7** | `search_messages` + `service_summary` + `service_summary` | 16 matches · 45,298 records |

**Flow:**  
**Question → DeepSeek → Python → Evidence → DeepSeek → ↻ → Final Answer**

> **207K+ logs stay outside the model context.**

### Final Answer

- Issue: libvirt image cache repeatedly flags the active base image 0673dd71 as unknown/removable and removes it — Service: nova.virt.libvirt.imagecache (path /var/lib/nova/instances/_base/a489c868...) — Frequency: 3,059 warnings, 2,064 "Removing base or swap file" (12,043 check cycles) — Evidence: "Unknown base file" / "Removable base files" paired with active-image "in use" logs.
- Issue: periodic image-cache pass crashes with InvalidDiskInfo — Service: oslo_service.periodic_task (ComputeManager._run_image_cache_manager_pass through nova.virt.libvirt) — Frequency: 8 — Evidence: qemu-img "No such file or directory" on just-deleted instance disks (16797a0f-32fb.../disk, daf38630..., 74e32610...) in the `_list_backing_images` race, "Exit code: 1".
- Issue: DB/hypervisor power-state mismatch — Service: nova.compute.manager — Frequency: 45 ("found 1 instances in the database and 0 on the hypervisor"; plus "sync_power_state ... pending task (spawning). Skip" ×3) — Evidence: lines 1297/5385/6773, node cp-1.slowvm1.tcloud-pg0.utah.cloudlab.us.
- Lower-priority: nova.virt.libvirt.driver vcpu-count warnings "cpu affinity is not supported" (12) and keystonemiddleware.auth_token token-validation 503s against identity server (2, 1 per day).

Overall: the dominant recurring problem is the libvirt image-cache manager thrashing on a single base image and crashing during its periodic pass, compounded by a stale compute-node DB state.
